In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score


In [17]:

strategies = [
    "fivecrop",
    "random5",
    "multiscale"
]

base_dir = "saved_models_multicrop"

seeds = [0,1,2,3,4]

results = []

def recall_at_k(sim_matrix, query_labels, gallery_labels, k=1):
    correct = 0

    for i in range(len(query_labels)):
        sims = sim_matrix[i]
        idx = np.argsort(-sims)[:k]

        if query_labels[i] in gallery_labels[idx]:
            correct += 1

    return correct / len(query_labels)



In [ ]:

for strategy in strategies:

    data = np.load(f"{base_dir}/{strategy}/data.npz")

    X = data["embeddings"]
    y = data["label_ids"]
    print("Strategy:", strategy)
    print("Embeddings shape:", X.shape)
    print("Labels shape:", y.shape)
    for seed in seeds:

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.2,
            # stratify=y, # STRATIFIKACIA
            random_state=seed
        )

        sim = cosine_similarity(X_test, X_train)

        # nearest neighbor prediction
        nn_idx = np.argmax(sim, axis=1)
        y_pred = y_train[nn_idx]

        acc = accuracy_score(y_test, y_pred)

        r1 = recall_at_k(sim, y_test, y_train, k=1)
        r5 = recall_at_k(sim, y_test, y_train, k=5)

        results.append({
            "strategy": strategy,
            "seed": seed,
            "accuracy": acc,
            "recall@1": r1,
            "recall@5": r5
        })



Strategy: fivecrop
Embeddings shape: (319, 768)
Labels shape: (319,)
Strategy: random5
Embeddings shape: (319, 768)
Labels shape: (319,)
Strategy: multiscale
Embeddings shape: (319, 768)
Labels shape: (319,)


In [19]:

df = pd.DataFrame(results)
df.to_csv("multicrop_results.csv", index=False)
summary = df.groupby("strategy")[["accuracy","recall@1","recall@5"]].mean()

print("\nResults per split:")
display(df.sort_values("accuracy", ascending=False))

print("\nAverage performance:")
display(summary.sort_values("accuracy", ascending=False))



Results per split:


,strategy,seed,accuracy,recall@1,recall@5
3,fivecrop,3,0.687500,0.687500,0.828125
0,fivecrop,0,0.656250,0.656250,0.796875
1,fivecrop,1,0.656250,0.656250,0.843750
9,random5,4,0.656250,0.656250,0.687500
2,fivecrop,2,0.640625,0.640625,0.859375
5,random5,0,0.625000,0.625000,0.843750
8,random5,3,0.625000,0.625000,0.828125
4,fivecrop,4,0.609375,0.609375,0.906250
6,random5,1,0.609375,0.609375,0.765625
14,multiscale,4,0.609375,0.609375,0.671875



Average performance:


,accuracy,recall@1,recall@5
strategy,,,
fivecrop,0.650000,0.650000,0.846875
random5,0.609375,0.609375,0.781250
multiscale,0.578125,0.578125,0.756250


In [20]:
import pandas as pd

df = pd.read_csv("multicrop_results.csv")

summary = (
    df.groupby("strategy")
    .agg(
        mean_accuracy=("accuracy", "mean"),
        std_accuracy=("accuracy", "std"),
        mean_recall1=("recall@1", "mean"),
        std_recall1=("recall@1", "std"),
        mean_recall5=("recall@5", "mean"),
        std_recall5=("recall@5", "std"),
    )
    .reset_index()
)

display(summary.sort_values("mean_accuracy", ascending=False))

,strategy,mean_accuracy,std_accuracy,mean_recall1,std_recall1,mean_recall5,std_recall5
0,fivecrop,0.650000,0.028384,0.650000,0.028384,0.846875,0.040444
2,random5,0.609375,0.046875,0.609375,0.046875,0.781250,0.061516
1,multiscale,0.578125,0.029232,0.578125,0.029232,0.756250,0.051349
